# Task 3: Caching Experiment (Queries 3, 5)

- Caches a reduced `integrated_taxi_trips` view containing only the columns used by Queries 3 and 5.
- Compares Query 3 and Query 5 before and after caching.

## Connect to Spark and imports

In [1]:
import sys
from time import perf_counter

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.runner import (
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)
from analytics.benchmarking import measure_registered_query, rows_match

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-Caching-Queries3And5')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 15:18:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Register the uncached Gold table

In [2]:
spark.catalog.clearCache()
register_integrated_taxi_trips(spark)

assert not spark.catalog.isCached(INTEGRATED_TABLE_VIEW)
spark.table(INTEGRATED_TABLE_VIEW).printSchema()

26/09/17 15:18:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- pickup_date: string (nullable = true)
 |-- dropoff_date: string (nullable = true)
 |-- trip_id: long (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- dropoff_zone_name: string (nullable = true)
 |-- dropof

26/09/17 15:18:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Measure uncached execution

In [3]:
QUERY_NAMES = (
    'query_03_air_quality_vs_demand',
    'query_05_peak_travel_hours',
)
REPETITIONS = 3

uncached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

print('Uncached measurements complete.')

Uncached measurements complete.


## Uncached physical plan

In [4]:
execute_registered_query(
    spark,
    'query_03_air_quality_vs_demand',
).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- BroadcastNestedLoopJoin Cross BuildRight (17)
         :- HashAggregate (6)
         :  +- Exchange (5)
         :     +- HashAggregate (4)
         :        +- Project (3)
         :           +- Filter (2)
         :              +- Scan parquet  (1)
         +- BroadcastExchange (16)
            +- HashAggregate (15)
               +- Exchange (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- Exchange (11)
                           +- HashAggregate (10)
                              +- Project (9)
                                 +- Filter (8)
                                    +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [pu_datetime_hour#3222, average_value_pu#3239, date#3253]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(average_value_pu)]
ReadSc

## Cache a reduced shared input

- Caching the full Gold table exceeds available executor memory.
- Queries 3 and 5 need only the below three columns.

In [5]:
CACHE_COLUMNS = (
    'pickup_datetime',
    'pu_datetime_hour',
    'average_value_pu',
)

cache_input = spark.table(INTEGRATED_TABLE_VIEW).select(*CACHE_COLUMNS)
cache_input.createOrReplaceTempView(INTEGRATED_TABLE_VIEW)

cache_started_at = perf_counter()
spark.catalog.cacheTable(INTEGRATED_TABLE_VIEW)
cached_input_rows = spark.table(INTEGRATED_TABLE_VIEW).count()
cache_materialization_seconds = perf_counter() - cache_started_at

assert spark.catalog.isCached(INTEGRATED_TABLE_VIEW)

print(f'Cached input rows: {cached_input_rows}')
print(f'Cache materialization: {cache_materialization_seconds:.3f} seconds')

Cached input rows: 9417833
Cache materialization: 1.025 seconds


## Measure cached execution and verify results

In [6]:
cached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

for query_name in QUERY_NAMES:
    assert rows_match(
        uncached_results[query_name]['rows'],
        cached_results[query_name]['rows'],
    ), (
        f'Results differ for {query_name}'
    )

print('All cached results match the uncached results within float tolerance.')

All cached results match the uncached results within float tolerance.


## Cached physical plan

- `Scan In-memory table` confirms that Spark uses the cache.

In [7]:
execute_registered_query(
    spark,
    'query_03_air_quality_vs_demand',
).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (23)
+- Sort (22)
   +- Exchange (21)
      +- BroadcastNestedLoopJoin Cross BuildRight (20)
         :- HashAggregate (9)
         :  +- Exchange (8)
         :     +- HashAggregate (7)
         :        +- Filter (6)
         :           +- Scan In-memory table integrated_taxi_trips (1)
         :                 +- InMemoryRelation (2)
         :                       +- * Project (5)
         :                          +- * ColumnarToRow (4)
         :                             +- Scan parquet  (3)
         +- BroadcastExchange (19)
            +- HashAggregate (18)
               +- Exchange (17)
                  +- HashAggregate (16)
                     +- HashAggregate (15)
                        +- Exchange (14)
                           +- HashAggregate (13)
                              +- Filter (12)
                                 +- Scan In-memory table integrated_taxi_trips (10)
                                       +- InMemor

## Benchmark summary

In [8]:
summary_rows = []
for query_name in QUERY_NAMES:
    uncached_median = uncached_results[query_name]['median_seconds']
    cached_median = cached_results[query_name]['median_seconds']
    summary_rows.append({
        'query': query_name,
        'uncached_samples_seconds': [
            round(value, 3)
            for value in uncached_results[query_name]['samples_seconds']
        ],
        'uncached_median_seconds': round(uncached_median, 3),
        'cached_samples_seconds': [
            round(value, 3)
            for value in cached_results[query_name]['samples_seconds']
        ],
        'cached_median_seconds': round(cached_median, 3),
        'speedup_factor': round(uncached_median / cached_median, 2),
        'cache_materialization_seconds': round(cache_materialization_seconds, 3),
    })

summary_rows

[{'query': 'query_03_air_quality_vs_demand',
  'uncached_samples_seconds': [1.181, 0.865, 0.745],
  'uncached_median_seconds': 0.865,
  'cached_samples_seconds': [0.263, 0.222, 0.174],
  'cached_median_seconds': 0.222,
  'speedup_factor': 3.9,
  'cache_materialization_seconds': 1.025},
 {'query': 'query_05_peak_travel_hours',
  'uncached_samples_seconds': [0.876, 0.85, 0.829],
  'uncached_median_seconds': 0.85,
  'cached_samples_seconds': [0.544, 0.52, 0.561],
  'cached_median_seconds': 0.544,
  'speedup_factor': 1.56,
  'cache_materialization_seconds': 1.025}]

## Cleanup

Run this cell only after saving the summary and plan output.

In [9]:
spark.catalog.uncacheTable(INTEGRATED_TABLE_VIEW)
spark.catalog.clearCache()
spark.stop()